# OpenLux Hub — 02 Analyse Contexte Quotidien & Qualité Territoriale

Ce notebook démontre la puissance de la table consolidée **`gold_luxembourg_daily_context`** :
- Croisement des **8 signaux quotidiens du Luxembourg** (Météo, Qualité de l'Air, Crues, Trafic, Électricité, Carburants, Inflation, Jours Fériés).
- Évaluation en un clic de la **complétude et qualité des données** (`data_quality_grade`, `data_completeness_pct`, `is_interpolated`).
- Qualification géographique des **grèves transfrontalières** (LU, FR, BE, DE).
- Disponibilité des infrastructures de **mobilité** (parkings 24/7, bornes Chargy, Vel'OH!).

## 1. Chargement de `gold_luxembourg_daily_context`

La table consolide chaque jour civil pour chaque commune les indicateurs clés du territoire.

In [ ]:
import duckdb

con = duckdb.connect()

df_context = con.execute("""
    SELECT 
        observation_date,
        commune_name,
        canton_name,
        is_holiday,
        holiday_name,
        avg_temperature_c,
        total_precipitation_mm,
        max_aqi_index,
        max_water_alert_status,
        electricity_spot_eur_mwh,
        diesel_eur_l,
        data_quality_grade,
        data_completeness_pct,
        is_interpolated
    FROM 'data/gold/gold_luxembourg_daily_context.parquet'
    ORDER BY observation_date DESC, commune_name ASC
    LIMIT 10
""").df()

df_context

## 2. Audit de Complétude et Qualité en 1 Clic

OpenLux classe automatiquement chaque ligne selon un grade de qualité strict :
- **`EXCELLENT`** : Capteurs locaux directs disponibles.
- **`GOOD`** : Mesures consolidées avec interpolation territoriale cantonale/nationale.
- **`PARTIAL`** : Signaux partiels nécessitant attention.

In [ ]:
df_quality = con.execute("""
    SELECT 
        data_quality_grade,
        count(*) AS nb_lignes,
        round(avg(data_completeness_pct), 1) AS moyenne_completude_pct,
        sum(CASE WHEN is_interpolated THEN 1 ELSE 0 END) AS nb_interpoles
    FROM 'data/gold/gold_luxembourg_daily_context.parquet'
    GROUP BY data_quality_grade
""").df()

df_quality

## 3. Analyse des Grèves et Perturbations Transfrontalières

Les flux de perturbations qualifient systématiquement le pays d'impact (`country_code` : LU, FR, BE, DE) pour les travailleurs frontaliers et résidents.

In [ ]:
df_strikes = con.execute("""
    SELECT 
        country_code,
        country_name,
        title,
        published_at_utc,
        category
    FROM 'data/gold/gold_strikes_disruptions.parquet'
    ORDER BY published_at_utc DESC
""").df()

df_strikes

## 4. Offre de Stationnement et Mobilité Douce

Consolidation des horaires d'ouverture (`24/7`) et de la disponibilité en temps réel des parkings, stations de recharge et vélos en libre-service.

In [ ]:
df_parkings = con.execute("""
    SELECT 
        facility_type,
        count(*) AS total_installations,
        sum(total_capacity) AS capacite_totale,
        sum(available_spaces) AS places_disponibles
    FROM 'data/gold/gold_mobility_parkings.parquet'
    GROUP BY facility_type
""").df()

df_parkings